In [28]:
import numpy as np
from scipy.optimize import minimize

class OptimizationSolver:
    def __init__(self):
        # Problem parameters
        self.n = 5
        self.mu = np.array([
            8.6033358901938017e-01, 3.4256184594817283e+00, 6.4372981791719468e+00,
            9.5293344053619631e+00, 1.2645287223856643e+01, 1.5771284874815882e+01,
            1.8902409956860023e+01, 2.2036496727938566e+01, 2.5172446326646664e+01,
            2.8309642854452012e+01, 3.1447714637546234e+01, 3.4586424215288922e+01,
            3.7725612827776501e+01, 4.0865170330488070e+01, 4.4005017920830845e+01,
            4.7145097736761031e+01, 5.0285366337773652e+01, 5.3425790477394663e+01,
            5.6566344279821521e+01, 5.9707007305335459e+01, 6.2847763194454451e+01,
            6.5988598698490392e+01, 6.9129502973895256e+01, 7.2270467060308960e+01,
            7.5411483488848148e+01, 7.8552545984242926e+01, 8.1693649235601683e+01,
            8.4834788718042290e+01, 8.7975960552493220e+01, 9.1117161394464745e+01
        ])

        # Compute A values
        self.A = self.compute_A()

    def compute_A(self):
        """Compute A values based on mu"""
        return 2 * np.sin(self.mu) / (self.mu + np.sin(self.mu) * np.cos(self.mu))

    def compute_constraint_value(self, x):
        """Compute the detailed constraint value"""
        # Compute rho
        rho = np.zeros(30)
        for j in range(30):
            rho[j] = -(np.exp(-self.mu[j]**2 * np.sum(x**2)) +
                       sum(2 * (-1)**(ii-1) * np.exp(-self.mu[j]**2 * np.sum(x[ii-1:]**2)) for ii in range(2, self.n+1)) +
                       (-1)**self.n) / self.mu[j]**2

        # First term: cross-terms
        cross_term = sum(
            self.mu[i]**2 * self.mu[j]**2 * self.A[i] * self.A[j] * rho[i] * rho[j] *
            (np.sin(self.mu[i]+self.mu[j])/(self.mu[i]+self.mu[j]) + np.sin(self.mu[i]-self.mu[j])/(self.mu[i]-self.mu[j]))
            for i in range(30) for j in range(i+1, 30)
        )

        # Second term: single-index terms
        single_term = sum(
            self.mu[j]**4 * self.A[j]**2 * rho[j]**2 * (np.sin(2*self.mu[j])/(2*self.mu[j]) + 1)/2
            for j in range(30)
        )

        # Third term
        third_term = sum(
            self.mu[j]**2 * self.A[j] * rho[j] * (2*np.sin(self.mu[j])/self.mu[j]**3 - 2*np.cos(self.mu[j])/self.mu[j]**2)
            for j in range(30)
        )

        # Full constraint calculation
        constraint_val = cross_term + single_term - third_term + 2/15

        return {
            'total': constraint_val,
            'cross_term': cross_term,
            'single_term': single_term,
            'third_term': third_term,
            'constant_term': 2/15
        }

    def objective_function(self, x):
        """Objective function: sum of squared x values"""
        return np.sum(x**2)

    def solve_optimization(self, method='SLSQP', print_details=True):
        """Solve the optimization problem"""
        # Initial guess
        x0 = np.zeros(self.n)

        # Constraint function for SLSQP
        def constraint(x):
            constraint_val = self.compute_constraint_value(x)['total'] - 0.0001
            return -constraint_val  # Negate for SLSQP

        # Solve the optimization problem
        result = minimize(
            self.objective_function,
            x0,
            method=method,
            constraints={'type': 'ineq', 'fun': constraint},
            bounds=[(None, None)]*self.n
        )

        # Print detailed results
        if print_details:
            print("\n--- Optimization Results ---")
            print("Optimal x:", result.x)
            print("Objective Value:", result.fun)
            print("Optimization Success:", result.success)
            print("Status Message:", result.message)

            # Compute and print constraint details for the optimal solution
            constraint_details = self.compute_constraint_value(result.x)
            print("\n--- Constraint Breakdown ---")
            print(f"Cross Term:     {constraint_details['cross_term']}")
            print(f"Single Term:    {constraint_details['single_term']}")
            print(f"Third Term:     {-constraint_details['third_term']}")
            print(f"Constant Term:  {constraint_details['constant_term']}")
            print(f"Total Constraint Value: {constraint_details['total']}")
            print(f"Constraint Satisfied (≤ 0.0001): {constraint_details['total'] <= 0.0001}")

        return result

# Run the optimization
solver = OptimizationSolver()
result = solver.solve_optimization()


--- Optimization Results ---
Optimal x: [-0.70246961 -0.07695024 -0.81974088  0.45647573 -0.00112951]
Objective Value: 1.3797313686833625
Optimization Success: True
Status Message: Optimization terminated successfully

--- Constraint Breakdown ---
Cross Term:     4.604514465177726e-18
Single Term:    0.13038630466516335
Third Term:     -0.26361963807852506
Constant Term:  0.13333333333333333
Total Constraint Value: 9.999991997161617e-05
Constraint Satisfied (≤ 0.0001): True
